In [1]:
!pip install torchmetrics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.4/840.4 kB 7.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 24.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 15.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 21.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 10.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 MB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 15.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━

In [2]:
!unzip PySESM.zip
!mkdir results_1
!mkdir results_2
!mkdir results_3
!mkdir results_1/plots
!mkdir results_2/plots
!mkdir results_3/plots
!mkdir results_1/stats
!mkdir results_2/stats
!mkdir results_3/stats


Archive:  PySESM.zip
  inflating: PySESM/__init__.py      
   creating: PySESM/base_functions/
 extracting: PySESM/base_functions/__init__.py  
  inflating: PySESM/base_functions/Function.py  
   creating: PySESM/models/
 extracting: PySESM/models/__init__.py  
  inflating: PySESM/models/DictLayer.py  
  inflating: PySESM/models/ISTALayer.py  
   creating: PySESM/models/SESM/
 extracting: PySESM/models/SESM/__init__.py  
  inflating: PySESM/models/SESM/SESM.py  
   creating: PySESM/test_functions/
 extracting: PySESM/test_functions/__init__.py  
  inflating: PySESM/test_functions/MultivariateNormal.py  
   creating: PySESM/utils/
 extracting: PySESM/utils/__init__.py  
  inflating: PySESM/utils/linalg.py  


In [3]:
#!pip install wandb -qU

In [4]:
import torch
#import wandb
from torchmetrics import MeanSquaredError

import numpy as np
from scipy.stats import multivariate_normal

from PySESM.models.SESM.SESM import SESM_Model
from PySESM.base_functions.Function import GaussianFunctions
import pandas as pd

from google.colab import files
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go

## Login de Wandb

In [5]:
#wandb.login()

## Definicion de covarianzas no diagnonales

In [6]:

# Non-diagonal covariance
def generate_sigma_tensors():
    """
    Generates non-diagonal covariance tensors for 2D Gaussian distributions.

    Returns:
    tuple: Tuple containing three non-diagonal covariance tensors (sigma1, sigma2, sigma3).
    """
    e0 = torch.tensor([1.0, 1.0], dtype=torch.float32)
    e0 = e0 / e0.norm()

    def generate_sigma(rotation_angle, scaling_factors):
        rotation_matrix = torch.tensor([[np.cos(rotation_angle), -np.sin(rotation_angle)],
                                       [np.sin(rotation_angle), np.cos(rotation_angle)]], dtype=torch.float32)
        e = torch.mm(rotation_matrix, e0.unsqueeze(1))
        E = torch.cat((e0.unsqueeze(1), e), dim=1)
        D = torch.diag(torch.tensor(scaling_factors, dtype=torch.float32))
        return torch.mm(torch.mm(E, D), E.t())

    sigma1 = generate_sigma(np.pi/4, [0.4, 0.1])
    sigma2 = generate_sigma(np.pi/4, [0.05, 0.5])
    sigma3 = generate_sigma(np.pi/4, [0.2, 0.5])

    return sigma1, sigma2, sigma3


In [7]:
sigma1, sigma2, sigma3 = generate_sigma_tensors()

## Definicion de varianzas diagonales

In [8]:
sigma1_d = 0.15 * torch.eye(2)
sigma2_d = 0.2 * torch.eye(2)
sigma3_d = 0.3 *torch.eye(2)

## Crear Gaussianas

In [9]:

def generate_mesh_and_z(sigma1, sigma2, sigma3):
    """
    Generates a mesh grid and corresponding probability density values for a mixture
    of three 2D Gaussian distributions.

    Args:
    sigma1 (torch.Tensor): The covariance tensor for the first Gaussian distribution.
    sigma2 (torch.Tensor): The covariance tensor for the second Gaussian distribution.
    sigma3 (torch.Tensor): The covariance tensor for the third Gaussian distribution.

    Returns:
    tuple: Tuple containing mesh grid arrays (xx, yy) and the corresponding probability
    density values (zz).
    """
    N_points = 50
    xl = -2
    xr = 2

    x = np.linspace(xl, xr, N_points+1)
    xx, yy = np.meshgrid(x, x)

    X = torch.tensor(np.column_stack([xx.ravel(), yy.ravel()]), dtype=torch.float32)

    mu1 = torch.tensor([1, 1], dtype=torch.float32)
    mu2 = torch.tensor([1, -1], dtype=torch.float32)
    mu3 = torch.tensor([-1, -1], dtype=torch.float32)

    z1 = torch.tensor(multivariate_normal.pdf(X.numpy(), mu1.numpy(), sigma1.numpy()), dtype=torch.float32)
    z2 = torch.tensor(multivariate_normal.pdf(X.numpy(), mu2.numpy(), sigma2.numpy()), dtype=torch.float32)
    z3 = torch.tensor(multivariate_normal.pdf(X.numpy(), mu3.numpy(), sigma3.numpy()), dtype=torch.float32)

    zz = (z1 + z2 + z3).reshape(xx.shape)

    return xx, yy, zz


## Covarianza de gaussianas del experimento
  - 3 Diagonales
  - 2 Diagonales, 1 no diagonal
  - 1 diagonal, 2 no diagonales
  - 3 no diagonales

In [10]:
# 3 diag
xx, yy, zz = generate_mesh_and_z(sigma1_d, sigma2_d, sigma3_d)

# 2 diag, 1 no diag
xx_1, yy_1, zz_1 = generate_mesh_and_z(sigma1_d, sigma2_d, sigma3)

# 1 diag, 2 no diag
xx_2, yy_2, zz_2 = generate_mesh_and_z(sigma1_d, sigma2, sigma3)
# 3 no diag
xx_3, yy_3, zz_3 = generate_mesh_and_z(sigma1, sigma2, sigma3)

print("X: ", xx)
print("Y: ", yy)
print("Z: ", zz)

X:  [[-2.   -1.92 -1.84 ...  1.84  1.92  2.  ]
 [-2.   -1.92 -1.84 ...  1.84  1.92  2.  ]
 [-2.   -1.92 -1.84 ...  1.84  1.92  2.  ]
 ...
 [-2.   -1.92 -1.84 ...  1.84  1.92  2.  ]
 [-2.   -1.92 -1.84 ...  1.84  1.92  2.  ]
 [-2.   -1.92 -1.84 ...  1.84  1.92  2.  ]]
Y:  [[-2.   -2.   -2.   ... -2.   -2.   -2.  ]
 [-1.92 -1.92 -1.92 ... -1.92 -1.92 -1.92]
 [-1.84 -1.84 -1.84 ... -1.84 -1.84 -1.84]
 ...
 [ 1.84  1.84  1.84 ...  1.84  1.84  1.84]
 [ 1.92  1.92  1.92 ...  1.92  1.92  1.92]
 [ 2.    2.    2.   ...  2.    2.    2.  ]]
Z:  tensor([[1.8926e-02, 2.4447e-02, 3.0913e-02,  ..., 1.1193e-02, 7.8721e-03,
         5.3619e-03],
        [2.4447e-02, 3.1580e-02, 3.9932e-02,  ..., 1.6434e-02, 1.1557e-02,
         7.8721e-03],
        [3.0913e-02, 3.9932e-02, 5.0494e-02,  ..., 2.3367e-02, 1.6434e-02,
         1.1193e-02],
        ...,
        [1.4548e-07, 1.8792e-07, 2.3763e-07,  ..., 9.6119e-03, 6.0114e-03,
         3.6026e-03],
        [6.7493e-08, 8.7185e-08, 1.1024e-07,  ..., 6.0114e-

In [11]:

fig = go.Figure(data=[go.Surface(z=zz_3.numpy(), x=xx_3, y=yy_3)])
fig.update_layout(scene=dict(aspectmode='data'))
fig.update_layout(scene=dict(camera=dict(eye=dict(x=2, y=2, z=1))))

#fig.show()

# Configuracion y ejecución del modelo

In [12]:


def generate_uniform_sampling(total_points, n_samples=500, min_separation=1):
    """
    Generate uniform sampling indices with a minimum separation criterion.

    Args:
        total_points (int): Total number of data points.
        n_samples (int): Number of samples to generate (default is 500).
        min_separation (int): Minimum separation between selected indices (default is 1).

    Returns:
        list: List of selected indices.

    Example:
        sampled_indices = generate_uniform_sampling(1000, n_samples=200, min_separation=2)
    """

    selected_indexes = []
    while len(selected_indexes) < n_samples:
        random_index = np.random.randint(total_points)
        if all(abs(random_index - existing_index) >= min_separation for existing_index in selected_indexes):
            selected_indexes.append(random_index)
    return selected_indexes

def sample_data(x_values, y_values, z_values, sampled_indices):
    """
    Sample data based on selected indices.

    Args:
        x_values (array-like): X-axis values.
        y_values (array-like): Y-axis values.
        z_values (array-like): Z-axis values.
        sampled_indices (list): List of indices to sample.

    Returns:
        tuple: Tuple containing sampled X (features) and y (labels).

    Example:
        X, y = sample_data(x_values, y_values, z_values, sampled_indices)
    """

    sampled_x = torch.tensor(x_values[sampled_indices], dtype=torch.float32)
    sampled_y = torch.tensor(y_values[sampled_indices], dtype=torch.float32)
    X = torch.stack((sampled_x, sampled_y), dim=1)
    y = z_values[sampled_indices].clone().detach().to(dtype=torch.float32)
    return X, y

def build_model(n_samples, n_features, l_functions, eig_range, mu_range):
    """
    Build and configure the SESM (Sparse Evolutionary Structural Modeling) model.

    Args:
    - n_samples (int): Number of training samples.
    - n_features (int): Number of features.
    - l_functions (int): Number of Gaussian functions.

    Returns:
    SESM_Model: An instance of the SESM model.
    """
    gaussian_function = GaussianFunctions(n_features=n_features, n_functions=l_functions, eig_range=eig_range, mu_range=mu_range)
    model = SESM_Model(
        n_samples=n_samples,
        psi=gaussian_function
    )
    return model

def train_model(model, X, y, model_epochs, ista_epochs, ista_alpha, ista_lambd, dictionary_epochs, dictionary_alpha):
    """
    Train the SESM model using the specified training parameters.

    Args:
    - model (SESM_Model): The SESM model to be trained.
    - X (torch.Tensor): Input features.
    - y (torch.Tensor): Target values.
    - model_epochs (int): Number of epochs for training the overall model.
    - ista_epochs (int): Number of ISTA (Iterative Shrinkage-Thresholding Algorithm) epochs.
    - ista_alpha (float): ISTA alpha parameter.
    - ista_lambd (float): ISTA lambda parameter.
    - dictionary_epochs (int): Number of epochs for updating the dictionary.
    - dictionary_alpha (float): Dictionary update alpha parameter.

    Returns:
    None
    """

    model.fit(
        X=X,
        y=y,
        model_epochs=model_epochs,
        ista_epochs=ista_epochs,
        ista_alpha=ista_alpha,
        ista_lambd=ista_lambd,
        dictionary_epochs=dictionary_epochs,
        dictionary_alpha=dictionary_alpha
    )

def plot_scatter(x_values, y_values, z_values, Z, hypset, fngroup, iteration):
    """
    Plot a scatter plot for the original function and the surrogate model.

    Args:
        x_values (array-like): X-axis values.
        y_values (array-like): Y-axis values.
        z_values (array-like): Z-axis values for the original function.
        Z (array-like): Z-axis values for the surrogate model.
        hypset (int): Hypothesis set or configuration identifier.
        fngroup (int): Function group identifier.
        iteration (int): Iteration number.

    Returns:
        None: Saves the plot as an image file.

    Example:
        plot_scatter(x_values, y_values, z_values, Z, 3, 1 , 1)
    """
    fig = plt.figure(figsize=(13, 13))

    ax1 = fig.add_subplot(221, projection='3d')
    ax1.scatter(x_values, y_values, z_values, c=z_values)
    ax1.set_xlabel('X')
    ax1.set_ylabel('Y')
    ax1.set_zlabel('Z')
    ax1.set_title('Original Function')

    ax2 = fig.add_subplot(222, projection='3d')
    ax2.scatter(x_values, y_values, Z.clone().detach(), c=Z.clone().detach())
    ax2.set_xlabel('X')
    ax2.set_ylabel('Y')
    ax2.set_zlabel('Z')
    ax2.set_title('Surrogate Model')
    ax2.set_zlim(0)

    filename = f"results_{hypset}/plots/{fngroup}.{iteration}.png"
    plt.savefig(filename)

    plt.clf()

def evaluate_model(model, x_values, y_values, z_values, hypset, fngroup, iter, debug=True):
    """
    Evaluate the SESM model's performance on a dataset.

    Args:
    - model (SESM_Model): The trained SESM model.
    - x_values (Union[np.ndarray, list]): Input feature values along the x-axis.
    - y_values (Union[np.ndarray, list]): Input feature values along the y-axis.
    - z_values (Union[np.ndarray, list]): Target values.
    - hypset (str): Hypothesis set identifier.
    - fngroup (str): Function group identifier (Dataset).
    - iter (int): Iteration number.
    - debug (bool): Whether to generate and save debugging plots (default is True).

    Returns:
    Tuple[float, float]: Tuple containing time taken for evaluation (in minutes) and
    the Mean Squared Error (MSE) value.
    """
    #----------------PREDICTION------------
    x_tensor = torch.tensor(x_values)
    y_tensor = torch.tensor(y_values)
    XY = torch.cat((x_tensor.unsqueeze(1), y_tensor.unsqueeze(1)), dim=1)
    Z = model.predict(XY)

    #----------------- ANALYSIS------------
    time = model.time / 60
    mse = MeanSquaredError()
    mse(Z.clone().detach(), z_values)
    mse_value = mse.compute()

    if debug:
        plot_scatter(x_values, y_values, z_values, Z, hypset, fngroup, iter)

        df = pd.DataFrame({
            'Loss': model.losses
        })

        df.to_csv(f'results_{hypset}/stats/{iter}.csv', index=False)

        """for loss in model.losses:
          wandb.log({"loss": loss})"""



    return time, mse_value

def run_experiment(_x, _y, _z, hyperparams, fngroup, iter, debug=True):
    """
    Run a complete SESM experiment, including data sampling, model training, and evaluation.

    Args:
    - _x (np.ndarray): Input feature values along the x-axis.
    - _y (np.ndarray): Input feature values along the y-axis.
    - _z (np.ndarray): Target values.
    - hyperparams (dict): Hyperparameters for the experiment.
    - fngroup (str): Function group identifier.
    - iter (int): Iteration number.
    - debug (bool): Whether to generate and save debugging plots (default is True).

    Returns:
    Tuple[float, float]: Tuple containing time taken for the experiment (in minutes)
    and the Mean Squared Error (MSE) value.
    """
    x_values = _x.ravel()
    y_values = _y.ravel()
    z_values = _z.ravel()

    total_points = len(x_values)

    sampled_indices = generate_uniform_sampling(total_points)
    X, y = sample_data(x_values, y_values, z_values, sampled_indices)

    model = build_model(n_samples=hyperparams["n_samples"], n_features=hyperparams["n_features"], l_functions=hyperparams["l_functions"], eig_range=hyperparams["eig_range"], mu_range=hyperparams["mu_range"])

    model_epochs = hyperparams["m_epochs"]
    ista_epochs = hyperparams["h_epochs"]
    dictionary_epochs = hyperparams["dict_epochs"]

    ista_alpha = hyperparams["ista_alpha"]
    ista_lambd = hyperparams["ista_lambd"]
    dictionary_alpha = hyperparams["dictionary_alpha"]

    train_model(model, X, y, model_epochs, ista_epochs, ista_alpha, ista_lambd, dictionary_epochs, dictionary_alpha)
    time, mse_value = evaluate_model(model, x_values, y_values, z_values, hyperparams["hyp_set"],  fngroup, iter, debug)
    return time, mse_value

In [13]:

def plot_stats(directory, num_files):
    """
    Plot statistics for loss values from multiple CSV files.

    Args:
        directory (str): The directory containing CSV files.
        num_files (int): The number of CSV files to process.

    Returns:
        None: Displays an interactive plot and saves an HTML file.

    Note:
        Assumes that each CSV file contains loss values for the same number of epochs.

    """
    fig = px.scatter(title='Loss analysis')
    m_epochs_losses = []

    for i in range(num_files):
        file_path = f"{directory}/stats/{i}.csv"

        df = pd.read_csv(file_path)
        m_epochs_losses.append(df)

    merged_losses = pd.concat(m_epochs_losses, axis=1)

    # Compute mean, std, min, and max for each row
    summary_df = pd.DataFrame({
        'Mean': merged_losses.mean(axis=1),
        'Std': merged_losses.std(axis=1),
        'Min': merged_losses.min(axis=1),
        'Max': merged_losses.max(axis=1)
    })

    summary_df.to_csv(f'{directory}/stats/processed.csv', index=False)

    fig.add_scatter(
        x=summary_df.index,
        y=summary_df['Mean'],
        mode='lines+markers',
        error_y=dict(type='data', array=summary_df['Std']),
        name='Mean'
    )

    fig.add_scatter(
        x=summary_df.index,
        y=summary_df['Max'],
        mode='markers',
        name='Max'
    )

    fig.add_scatter(
        x=summary_df.index,
        y=summary_df['Min'],
        mode='markers',
        name='Min'
    )

    fig.update_layout(
        xaxis_title='m_epochs',
        yaxis_title='Loss',
        legend_title='Legend',
        showlegend=True
    )
    fig.show()
    fig.write_html(f"interactive_plot-{directory}.html")

In [14]:
import csv

def save_results(data, fngroup):
  # Compute Mean and Std for executio
  times = [item[1] for item in data]
  mse_values = [item[2] for item in data]

  average_time = np.mean(times)
  std_time = np.std(times)
  average_mse = np.mean(mse_values)
  std_mse = np.std(mse_values)

  with open(f"results_{fngroup}.csv", mode="w", newline="") as file:
      writer = csv.writer(file)
      writer.writerow(["Repetion", "Time (min)", "MSE"])
      writer.writerows(data)
      writer.writerow(["Mean", average_time, average_mse])
      writer.writerow(["Std", std_time, std_mse])


# Nomenclatura de experimentos

<Set de hiperparámetros>. <Set de datos (conjunto de gaussianas)>.<Número de repetición del experimento>


## Set de Hiperparámetros
|  Hiperparámetro | Exp 1.x.x     | Exp 2.x.x     | Exp 3.x.x     |
|-----------------|---------------|---------------|---------------|
| n_samples       | 50            | 100           | 500           |
| n_features      | 2             | 2             | 2             |
| l_functions     | 20            | 6             | 10            |
| ista_alpha      | 0.06          | 0.0125        | 0.0125        |
| ista_lambd      | 0.005         | 0.001         | 0.001         |
| dictionary_alpha| 0.06          | 0.0125        | 0.0125        |
| m_epochs        | 25            | 500           | 300           |
| dict_epochs     | 800           | 20            | 60            |
| h_epochs        | 1000          | 50            | 100           |


### Set de datos

|     Set      | Exp 1.x.x     | Exp 2.x.x     | Exp 3.x.x     |
|-----------------|---------------|---------------|---------------|
| Gaussianas 1    | Exp 1.1.x     | Exp 2.1.x     | Exp 3.1.x     |
| Gaussianas 2    | Exp 1.2.x     | Exp 2.2.x     | Exp 3.2.x     |
| Gaussianas 3    | Exp 1.3.x     | Exp 2.3.x     | Exp 3.3.x     |
| Gaussianas 4    | Exp 1.4.x     | Exp 2.4.x     | Exp 3.4.x     |


In [15]:
N_iter = 5 #
experiment_3 = {
      "hyp_set": 3,
      "n_samples"	: 500,
      "n_features" : 2,
      "l_functions":  10,
      "eig_range": [1e0, 1e1],
      "mu_range": [0, 1],
      "ista_alpha"	: 0.0125,
      "ista_lambd"	 : 0.001,
      "dictionary_alpha": 0.0125,
      "m_epochs" : 20,
      "dict_epochs": 60,
      "h_epochs": 100
}
#1, 2, 3 --> m_epochs


In [16]:
data = []
for i in range(N_iter):
  """wandb.init(
    # Set the project where this run will be logged
    project="SESM-2D-GaussianFunction",
    # We pass a run name (otherwise it’ll be randomly assigned, like sunshine-lollypop-10)
    name=f"12-11-2023-experiments-{i}",
    # Track hyperparameters and run metadata
    config = experiment_3
    )
"""
  time, mse = run_experiment(xx,yy,zz,
                            experiment_3, fngroup=1, iter=i)
  data.append((i, time, mse.item()))
#wandb.finish()
save_results(data=data, fngroup=1)

Epoch 1 Loss: 1.6682305335998535

Epoch 2 Loss: 0.4286251962184906

Epoch 3 Loss: 0.17192016541957855

Epoch 4 Loss: 0.09271705895662308

Epoch 5 Loss: 0.06367533653974533

Epoch 6 Loss: 0.05134929344058037

Epoch 7 Loss: 0.04520639404654503

Epoch 8 Loss: 0.04160642251372337

Epoch 9 Loss: 0.03918861597776413

Epoch 10 Loss: 0.037382736802101135

Epoch 11 Loss: 0.03592249006032944

Epoch 12 Loss: 0.03467528522014618

Epoch 13 Loss: 0.03356490656733513

Epoch 14 Loss: 0.03252769634127617

Epoch 15 Loss: 0.031536757946014404

Epoch 16 Loss: 0.030574612319469452

Epoch 17 Loss: 0.0296302679926157

Epoch 18 Loss: 0.02869744785130024

Epoch 19 Loss: 0.027773519977927208

Epoch 20 Loss: 0.026858650147914886

Epoch 1 Loss: 1.3672552108764648

Epoch 2 Loss: 0.4739847481250763

Epoch 3 Loss: 0.23181192576885223

Epoch 4 Loss: 0.13979806005954742

Epoch 5 Loss: 0.0983182042837143

Epoch 6 Loss: 0.07718711346387863

Epoch 7 Loss: 0.06521222740411758

Epoch 8 Loss: 0.05773207172751427

Epoch 9 Lo

<Figure size 1300x1300 with 0 Axes>

<Figure size 1300x1300 with 0 Axes>

<Figure size 1300x1300 with 0 Axes>

<Figure size 1300x1300 with 0 Axes>

<Figure size 1300x1300 with 0 Axes>

In [17]:
#Unit test de la inicialización únicamente
stats_dir = f'results_{experiment_3["hyp_set"]}'
plot_stats(stats_dir, N_iter)

# Correr experimentos

In [18]:
data_1 = []

for i in range(N_iter):
  """wandb.init(
    # Set the project where this run will be logged
    project="SESM-2D-GaussianFunction",
    # We pass a run name (otherwise it’ll be randomly assigned, like sunshine-lollypop-10)
    name=f"12-11-2023-experiments-{i}",
    # Track hyperparameters and run metadata
    config = experiment_3
    )
"""
  time, mse = run_experiment(xx_1,yy_1,zz_1,
                            experiment_3, fngroup=2, iter=i)
  data_1.append((i, time, mse.item()))

save_results(data=data_1, fngroup=2)

Epoch 1 Loss: 1.7977228164672852

Epoch 2 Loss: 0.47265294194221497

Epoch 3 Loss: 0.19408605992794037

Epoch 4 Loss: 0.10415202379226685

Epoch 5 Loss: 0.06978357583284378

Epoch 6 Loss: 0.05492229759693146

Epoch 7 Loss: 0.0476728156208992

Epoch 8 Loss: 0.04365575313568115

Epoch 9 Loss: 0.04115316644310951

Epoch 10 Loss: 0.039428818970918655

Epoch 11 Loss: 0.03814137354493141

Epoch 12 Loss: 0.03711877390742302

Epoch 13 Loss: 0.03626693785190582

Epoch 14 Loss: 0.03553065285086632

Epoch 15 Loss: 0.034875642508268356

Epoch 16 Loss: 0.034279659390449524

Epoch 17 Loss: 0.03372778370976448

Epoch 18 Loss: 0.03320971876382828

Epoch 19 Loss: 0.03271814063191414

Epoch 20 Loss: 0.03224775567650795

Epoch 1 Loss: 1.894263744354248

Epoch 2 Loss: 0.47549596428871155

Epoch 3 Loss: 0.18385832011699677

Epoch 4 Loss: 0.10089568048715591

Epoch 5 Loss: 0.07247522473335266

Epoch 6 Loss: 0.06051473692059517

Epoch 7 Loss: 0.05423804000020027

Epoch 8 Loss: 0.05023809149861336

Epoch 9 Lo

<Figure size 1300x1300 with 0 Axes>

<Figure size 1300x1300 with 0 Axes>

<Figure size 1300x1300 with 0 Axes>

<Figure size 1300x1300 with 0 Axes>

<Figure size 1300x1300 with 0 Axes>

In [19]:
data_2 = []
for i in range(N_iter):
  """  wandb.init(
    # Set the project where this run will be logged
    project="SESM-2D-GaussianFunction",
    # We pass a run name (otherwise it’ll be randomly assigned, like sunshine-lollypop-10)
    name=f"12-11-2023-experiments-{i}",
    # Track hyperparameters and run metadata
    config = experiment_3
    )
"""
  time, mse = run_experiment(xx_2,yy_2,zz_2,
                            experiment_3,fngroup=3, iter=i)
  data_2.append((i, time, mse.item()))

save_results(data=data_2, fngroup=3)

Epoch 1 Loss: 1.221996545791626

Epoch 2 Loss: 0.4670615494251251

Epoch 3 Loss: 0.2403389811515808

Epoch 4 Loss: 0.14788351953029633

Epoch 5 Loss: 0.10461557656526566

Epoch 6 Loss: 0.08258959650993347

Epoch 7 Loss: 0.07061412930488586

Epoch 8 Loss: 0.06369977444410324

Epoch 9 Loss: 0.0594102181494236

Epoch 10 Loss: 0.056564897298812866

Epoch 11 Loss: 0.054559126496315

Epoch 12 Loss: 0.05306060239672661

Epoch 13 Loss: 0.0518791563808918

Epoch 14 Loss: 0.05090165510773659

Epoch 15 Loss: 0.050058480352163315

Epoch 16 Loss: 0.049305595457553864

Epoch 17 Loss: 0.048617083579301834

Epoch 18 Loss: 0.04798009991645813

Epoch 19 Loss: 0.04737364128232002

Epoch 20 Loss: 0.0467965342104435

Epoch 1 Loss: 1.3651176691055298

Epoch 2 Loss: 0.45034706592559814

Epoch 3 Loss: 0.21411679685115814

Epoch 4 Loss: 0.12482397258281708

Epoch 5 Loss: 0.08544851839542389

Epoch 6 Loss: 0.0666966661810875

Epoch 7 Loss: 0.0572742335498333

Epoch 8 Loss: 0.05224642902612686

Epoch 9 Loss: 0.0

<Figure size 1300x1300 with 0 Axes>

<Figure size 1300x1300 with 0 Axes>

<Figure size 1300x1300 with 0 Axes>

<Figure size 1300x1300 with 0 Axes>

<Figure size 1300x1300 with 0 Axes>

In [20]:
files.download('results_1.csv')
!zip -r results_1.zip results_1/
files.download('results_1.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  adding: results_1/ (stored 0%)
  adding: results_1/stats/ (stored 0%)
  adding: results_1/plots/ (stored 0%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
files.download('results_2.csv')
!zip -r results_2.zip results_2/
files.download('results_2.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  adding: results_2/ (stored 0%)
  adding: results_2/stats/ (stored 0%)
  adding: results_2/plots/ (stored 0%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
files.download('results_3.csv')
!zip -r results_3.zip results_3/
files.download('results_3.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  adding: results_3/ (stored 0%)
  adding: results_3/stats/ (stored 0%)
  adding: results_3/stats/1.csv (deflated 49%)
  adding: results_3/stats/2.csv (deflated 49%)
  adding: results_3/stats/4.csv (deflated 49%)
  adding: results_3/stats/processed.csv (deflated 51%)
  adding: results_3/stats/0.csv (deflated 48%)
  adding: results_3/stats/3.csv (deflated 49%)
  adding: results_3/plots/ (stored 0%)
  adding: results_3/plots/2.1.png (deflated 4%)
  adding: results_3/plots/1.2.png (deflated 4%)
  adding: results_3/plots/3.2.png (deflated 4%)
  adding: results_3/plots/3.3.png (deflated 4%)
  adding: results_3/plots/2.4.png (deflated 4%)
  adding: results_3/plots/1.1.png (deflated 4%)
  adding: results_3/plots/1.4.png (deflated 5%)
  adding: results_3/plots/1.0.png (deflated 5%)
  adding: results_3/plots/2.3.png (deflated 4%)
  adding: results_3/plots/3.0.png (deflated 5%)
  adding: results_3/plots/2.2.png (deflated 4%)
  adding: results_3/plots/2.0.png (deflated 4%)
  adding: results_3/plo

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>